# Frankfurt Bike-Sharing: GCN Loader

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/data_loaders/frankfurt_bike_sharing/notebooks/frankfurt_bike_sharing_gcn_loader.ipynb)

This notebook builds a small GCN-ready station graph for Frankfurt from the TUM FTM European Bike-Sharing Dataset sample files.


In [ ]:
!pip -q install pandas folium


In [ ]:
from pathlib import Path
import math
import urllib.request

import folium
import pandas as pd

pd.set_option("display.max_columns", 80)


## 1. Download sample CSV files

The full dataset is large. The notebook starts with the source repository's small `sample/` files. Use locally downloaded full CSV files with the same names when you need trip-derived edges for Frankfurt.


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/TUMFTM/european-bike-sharing-dataset/main/sample"
FILES = ["cities.csv", "stations.csv", "trips.csv", "station_status.csv"]
DATA_DIR = Path("/content/frankfurt_bike_sharing_sample")
DATA_DIR.mkdir(parents=True, exist_ok=True)

for filename in FILES:
    target = DATA_DIR / filename
    if not target.exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{filename}", target)

cities = pd.read_csv(DATA_DIR / "cities.csv")
stations = pd.read_csv(DATA_DIR / "stations.csv")
trips = pd.read_csv(DATA_DIR / "trips.csv")
station_status = pd.read_csv(DATA_DIR / "station_status.csv")

frankfurt = cities[(cities["name"] == "Frankfurt") & (cities["country"] == "DE")].iloc[0]
CITY_ID = int(frankfurt["id"])
CITY_ID


## 2. Build node features

Nodes are Frankfurt stations. Features include coordinates, rack counts, and simple station-status summaries when available.


In [ ]:
frankfurt_stations = stations[stations["city_id"] == CITY_ID].copy()
frankfurt_stations = frankfurt_stations.sort_values("id").reset_index(drop=True)
frankfurt_stations["node_index"] = range(len(frankfurt_stations))

status = station_status.merge(
    frankfurt_stations[["id"]].rename(columns={"id": "station_id"}),
    on="station_id",
    how="inner",
)
status_features = (
    status.groupby("station_id", as_index=False)
    .agg(
        status_observations=("time", "count"),
        avg_bikes=("bikes", "mean"),
        avg_bikes_available_to_rent=("bikes_available_to_rent", "mean"),
        avg_free_racks=("free_racks", "mean"),
    )
    if len(status)
    else pd.DataFrame(columns=["station_id", "status_observations", "avg_bikes", "avg_bikes_available_to_rent", "avg_free_racks"])
)

nodes = frankfurt_stations.merge(status_features, left_on="id", right_on="station_id", how="left")
for col in ["status_observations", "avg_bikes", "avg_bikes_available_to_rent", "avg_free_racks"]:
    nodes[col] = nodes[col].fillna(0)

nodes = nodes[
    [
        "node_index",
        "id",
        "name",
        "city_id",
        "lon",
        "lat",
        "bike_racks",
        "special_racks",
        "terminal_type",
        "place_type",
        "status_observations",
        "avg_bikes",
        "avg_bikes_available_to_rent",
        "avg_free_racks",
    ]
].rename(columns={"id": "station_id"})

print(f"Frankfurt stations: {len(nodes)}")
display(nodes.head())


## 3. Build edges

If Frankfurt trip rows are available, edges are trip counts between stations. In the small sample there may be no Frankfurt trips, so the notebook falls back to spatial k-nearest-neighbor station edges.


In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    radius_m = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return radius_m * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

station_to_node = dict(zip(nodes["station_id"], nodes["node_index"]))
trip_edges = trips[
    (trips["city_id"] == CITY_ID)
    & trips["station_id_start"].notna()
    & trips["station_id_end"].notna()
    & (trips["station_id_start"] != trips["station_id_end"])
].copy()
trip_edges = trip_edges[
    trip_edges["station_id_start"].isin(station_to_node)
    & trip_edges["station_id_end"].isin(station_to_node)
]

if len(trip_edges):
    edges = (
        trip_edges.groupby(["station_id_start", "station_id_end"], as_index=False)
        .agg(trip_count=("bike_id", "count"), avg_distance_m=("distance", "mean"), avg_duration_s=("duration", "mean"))
        .rename(columns={"station_id_start": "source_station_id", "station_id_end": "target_station_id"})
    )
    edges["source"] = edges["source_station_id"].map(station_to_node)
    edges["target"] = edges["target_station_id"].map(station_to_node)
    edges["edge_type"] = "trip"
    edges["edge_weight"] = edges["trip_count"]
else:
    K = 4
    edge_rows = {}
    node_records = nodes.to_dict("records")
    for source in node_records:
        distances = []
        for target in node_records:
            if source["node_index"] == target["node_index"]:
                continue
            distance_m = haversine_m(source["lon"], source["lat"], target["lon"], target["lat"])
            distances.append((distance_m, target))
        for distance_m, target in sorted(distances, key=lambda item: item[0])[:K]:
            a, b = sorted([source["node_index"], target["node_index"]])
            edge_rows[(a, b)] = {
                "source": a,
                "target": b,
                "source_station_id": nodes.loc[nodes["node_index"] == a, "station_id"].iloc[0],
                "target_station_id": nodes.loc[nodes["node_index"] == b, "station_id"].iloc[0],
                "edge_type": "spatial_knn",
                "edge_weight": 1 / (1 + distance_m / 1000),
                "trip_count": 0,
                "avg_distance_m": distance_m,
                "avg_duration_s": 0,
            }
    edges = pd.DataFrame(edge_rows.values())

edges = edges[["source", "target", "source_station_id", "target_station_id", "edge_type", "edge_weight", "trip_count", "avg_distance_m", "avg_duration_s"]]
print(edges["edge_type"].value_counts())
display(edges.head())


## 4. Save GCN tables and map the graph


In [ ]:
OUTPUT_DIR = Path("/content/frankfurt_bike_sharing_gcn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
nodes_path = OUTPUT_DIR / "frankfurt_bike_sharing_gcn_nodes.csv"
edges_path = OUTPUT_DIR / "frankfurt_bike_sharing_gcn_edges.csv"
nodes.to_csv(nodes_path, index=False)
edges.to_csv(edges_path, index=False)
print(nodes_path)
print(edges_path)

m = folium.Map(location=[nodes["lat"].mean(), nodes["lon"].mean()], zoom_start=12, tiles="cartodbpositron")
node_lookup = nodes.set_index("node_index")
for _, edge in edges.iterrows():
    a = node_lookup.loc[edge["source"]]
    b = node_lookup.loc[edge["target"]]
    folium.PolyLine([(a["lat"], a["lon"]), (b["lat"], b["lon"])], color="#777", weight=1, opacity=0.5).add_to(m)
for _, row in nodes.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        fill=True,
        color="#1f78b4",
        fill_opacity=0.8,
        tooltip=f"{row['name']} ({row['station_id']})",
    ).add_to(m)
m
